<a href="https://colab.research.google.com/github/Fatma3598/elsewedy-demand-forecasting/blob/main/ElSewedy_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# ============================================================================
# CONFIGURATION
# ============================================================================

START_DATE = '2022-01-01'
END_DATE = '2024-12-31'
XLPE_GROWTH_RATE = 0.20  # XLPE has highest demand growth
SEMICONDUCTOR_GROWTH_RATE = 0.19
DEFAULT_GROWTH_RATE = 0.10

# Material definitions based on problem statement
MATERIALS = [
    # Polymer layers - XLPE (highest demand)
    {'code': 'XLPE-11KV-INS', 'desc': 'XLPE Insulation 11KV', 'type': 'XLPE', 'voltage': '11KV',
     'base_demand': 45, 'growth': 0.20, 'seasonal': True, 'category': 'B'},
    {'code': 'XLPE-33KV-INS', 'desc': 'XLPE Insulation 33KV', 'type': 'XLPE', 'voltage': '33KV',
     'base_demand': 35, 'growth': 0.18, 'seasonal': True, 'category': 'B'},
    {'code': 'XLPE-66KV-INS', 'desc': 'XLPE Insulation 66KV', 'type': 'XLPE', 'voltage': '66KV',
     'base_demand': 28, 'growth': 0.17, 'seasonal': True, 'category': 'B'},

    # Polymer layers - PE
    {'code': 'PE-LV-JKT', 'desc': 'PE Jacket Low Voltage', 'type': 'PE', 'voltage': 'LV',
     'base_demand': 25, 'growth': 0.10, 'seasonal': False, 'category': 'B'},
    {'code': 'PE-MV-INS', 'desc': 'PE Insulation Medium Voltage', 'type': 'PE', 'voltage': 'MV',
     'base_demand': 20, 'growth': 0.09, 'seasonal': False, 'category': 'B'},

    # Polymer layers - PVC
    {'code': 'PVC-LV-INS', 'desc': 'PVC Insulation Low Voltage', 'type': 'PVC', 'voltage': 'LV',
     'base_demand': 30, 'growth': 0.08, 'seasonal': False, 'category': 'B'},
    {'code': 'PVC-LV-JKT', 'desc': 'PVC Jacket Low Voltage', 'type': 'PVC', 'voltage': 'LV',
     'base_demand': 32, 'growth': 0.07, 'seasonal': False, 'category': 'B'},

    # Polymer layers - LSF
    {'code': 'LSF-LV-JKT', 'desc': 'LSF Jacket Low Voltage', 'type': 'LSF', 'voltage': 'LV',
     'base_demand': 15, 'growth': 0.12, 'seasonal': False, 'category': 'B'},

    # Shielding layers - GSW
    {'code': 'GSW-MV-SHLD', 'desc': 'Galvanized Steel Wire Shield MV', 'type': 'GSW', 'voltage': 'MV',
     'base_demand': 20, 'growth': 0.10, 'seasonal': False, 'category': 'B'},
    {'code': 'GSW-HV-SHLD', 'desc': 'Galvanized Steel Wire Shield HV', 'type': 'GSW', 'voltage': 'HV',
     'base_demand': 16, 'growth': 0.09, 'seasonal': False, 'category': 'B'},

    # Shielding layers - GST
    {'code': 'GST-HV-TAPE', 'desc': 'Galvanized Steel Tape HV', 'type': 'GST', 'voltage': 'HV',
     'base_demand': 18, 'growth': 0.11, 'seasonal': False, 'category': 'B'},

    # Shielding layers - Copper/Aluminum Screen
    {'code': 'CU-SCR-11KV', 'desc': 'Copper Screen Tape 11KV', 'type': 'Copper Screen', 'voltage': '11KV',
     'base_demand': 12, 'growth': 0.09, 'seasonal': False, 'category': 'A'},
    {'code': 'AL-SCR-33KV', 'desc': 'Aluminum Screen Tape 33KV', 'type': 'Aluminum Screen', 'voltage': '33KV',
     'base_demand': 10, 'growth': 0.08, 'seasonal': False, 'category': 'A'},

    # Screening/Blocking materials
    {'code': 'MICA-TAPE-HV', 'desc': 'Mica Tape High Voltage', 'type': 'Mica Tape', 'voltage': 'HV',
     'base_demand': 8, 'growth': 0.07, 'seasonal': False, 'category': 'B'},
    {'code': 'WB-TAPE-MV', 'desc': 'Water Blocking Tape MV', 'type': 'Water Blocking Tape', 'voltage': 'MV',
     'base_demand': 22, 'growth': 0.13, 'seasonal': True, 'category': 'B'},

    # Semiconductors (high demand mentioned in notes)
    {'code': 'SEMI-11KV-LAY', 'desc': 'Semiconductor Layer 11KV', 'type': 'Semiconductor', 'voltage': '11KV',
     'base_demand': 28, 'growth': 0.19, 'seasonal': True, 'category': 'B'},
    {'code': 'SEMI-33KV-LAY', 'desc': 'Semiconductor Layer 33KV', 'type': 'Semiconductor', 'voltage': '33KV',
     'base_demand': 24, 'growth': 0.18, 'seasonal': True, 'category': 'B'},
]

# Supplier database
SUPPLIERS = [
    {'name': 'PolyTech Europe', 'country': 'Germany', 'specialty': ['XLPE', 'PE', 'PVC'], 'performance': 9, 'base_lead_time': 35},
    {'name': 'ChinaPoly Industries', 'country': 'China', 'specialty': ['PVC', 'PE', 'LSF'], 'performance': 7, 'base_lead_time': 45},
    {'name': 'Nordic Cables Supply', 'country': 'Sweden', 'specialty': ['XLPE', 'Semiconductor'], 'performance': 10, 'base_lead_time': 30},
    {'name': 'TurkCable Materials', 'country': 'Turkey', 'specialty': ['PVC', 'LSF', 'GSW'], 'performance': 8, 'base_lead_time': 25},
    {'name': 'MetalShield Corp', 'country': 'South Korea', 'specialty': ['GSW', 'GST', 'Copper Screen', 'Aluminum Screen'], 'performance': 9, 'base_lead_time': 40},
    {'name': 'IndiaScreen Tech', 'country': 'India', 'specialty': ['Aluminum Screen', 'Mica Tape', 'Water Blocking Tape'], 'performance': 7, 'base_lead_time': 50},
    {'name': 'USATape Solutions', 'country': 'USA', 'specialty': ['Mica Tape', 'Water Blocking Tape'], 'performance': 8, 'base_lead_time': 42},
    {'name': 'JapanSemi Advanced', 'country': 'Japan', 'specialty': ['Semiconductor', 'XLPE'], 'performance': 10, 'base_lead_time': 38},
]

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def get_suppliers_for_material(material_type):
    """Get suitable suppliers for a material type"""
    suitable = [s for s in SUPPLIERS if material_type in s['specialty']]
    return suitable if suitable else SUPPLIERS[:3]

def calculate_demand(base_demand, growth_rate, months_elapsed, seasonal, month):
    """Calculate demand with growth trend and seasonality"""
    # Base trend with growth
    trend = base_demand * (1 + growth_rate) ** (months_elapsed / 12)

    # Add seasonality (higher in Q2 and Q4 for infrastructure projects)
    if seasonal:
        seasonal_factor = 1 + 0.15 * np.sin(2 * np.pi * month / 12 + np.pi/2)
        trend *= seasonal_factor

    # Add random noise (±10%)
    noise = np.random.normal(1, 0.10)
    demand = max(0, trend * noise)

    return round(demand, 2)

def calculate_safety_stock(avg_demand, lead_time, risk_score):
    """Calculate safety stock based on demand variability and risk"""
    # Higher risk = higher safety stock multiplier
    risk_multiplier = 1 + (risk_score / 10) * 0.5
    # Lead time coverage (typically 1.5-2x lead time demand)
    lead_time_demand = avg_demand * (lead_time / 30)
    safety_stock = lead_time_demand * risk_multiplier
    return round(safety_stock, 2)

def calculate_risk_score(num_suppliers, lead_time, performance_score):
    """Calculate supply chain risk score (1-10, higher = more risk)"""
    # Fewer suppliers = higher risk
    supplier_risk = max(1, 11 - num_suppliers * 2)
    # Longer lead time = higher risk
    lead_time_risk = min(10, lead_time / 7)
    # Lower performance = higher risk
    performance_risk = 11 - performance_score

    # Weighted average
    risk = (supplier_risk * 0.4 + lead_time_risk * 0.3 + performance_risk * 0.3)
    return round(np.clip(risk, 1, 10), 1)

def get_payment_terms(category, supplier_performance):
    """Get payment terms based on material category and supplier relationship"""
    terms_options = {
        'A': ['Net 90', 'Net 120', 'Annual Contract'],  # Metals - longer terms
        'B': ['Net 30', 'Net 45', 'Net 60'],  # Non-metals
        'C': ['Net 30', 'Net 45']
    }

    # Better suppliers get better terms
    if supplier_performance >= 9:
        return random.choice(terms_options.get(category, ['Net 30']))
    else:
        return random.choice(terms_options.get(category, ['Net 30'])[:2])

# ============================================================================
# MAIN DATASET GENERATION
# ============================================================================

def generate_dataset():
    """Generate complete synthetic dataset"""

    print("Generating Elsewedy Cable Materials Dataset...")
    print(f"Date Range: {START_DATE} to {END_DATE}")
    print(f"Materials: {len(MATERIALS)}")

    # Date range
    start = datetime.strptime(START_DATE, '%Y-%m-%d')
    end = datetime.strptime(END_DATE, '%Y-%m-%d')

    # Generate monthly records
    data = []
    current_date = start

    while current_date <= end:
        month_num = current_date.month
        months_elapsed = (current_date.year - start.year) * 12 + (current_date.month - start.month)

        for material in MATERIALS:
            # Get suppliers for this material
            suitable_suppliers = get_suppliers_for_material(material['type'])
            num_alt_suppliers = len(suitable_suppliers)

            # Select primary supplier (weighted by performance)
            weights = [s['performance'] for s in suitable_suppliers]
            supplier = random.choices(suitable_suppliers, weights=weights, k=1)[0]

            # Calculate demand
            demand = calculate_demand(
                material['base_demand'],
                material['growth'],
                months_elapsed,
                material['seasonal'],
                month_num
            )

            # Lead time with some variability
            lead_time = supplier['base_lead_time'] + random.randint(-5, 5)

            # Calculate risk score
            risk_score = calculate_risk_score(
                num_alt_suppliers,
                lead_time,
                supplier['performance']
            )

            # Calculate safety stock
            safety_stock = calculate_safety_stock(
                material['base_demand'],
                lead_time,
                risk_score
            )

            # Current stock level (simulate inventory dynamics)
            # Stock should cover ~2-3 months of demand
            stock_target = demand * random.uniform(2, 3)
            current_stock = stock_target + np.random.normal(0, stock_target * 0.15)
            current_stock = max(0, current_stock)

            # Units sold/used (slightly less than demand with some variation)
            units_used = demand * random.uniform(0.85, 1.0)

            # Price per unit (with some market fluctuation)
            base_prices = {
                'XLPE': 3.5, 'PE': 2.8, 'PVC': 2.2, 'LSF': 3.0,
                'GSW': 4.2, 'GST': 3.8, 'Copper Screen': 8.5, 'Aluminum Screen': 5.5,
                'Mica Tape': 6.0, 'Water Blocking Tape': 2.5, 'Semiconductor': 7.5
            }
            base_price = base_prices.get(material['type'], 3.0)
            price = base_price * (1 + np.random.normal(0, 0.08))  # ±8% variation

            # Market trends index (100 = baseline, simulates infrastructure demand)
            # Higher in Q2 and Q4
            market_trend = 100 + 10 * np.sin(2 * np.pi * month_num / 12) + np.random.normal(0, 5)

            # Generate PO number
            po_number = f"PO-{current_date.year}-{material['code'][:4]}-{random.randint(1000,9999)}"
            doc_number = f"DOC-{random.randint(100000,999999)}"

            # Create record
            record = {
                'Date': current_date.strftime('%Y-%m-%d'),
                'Item_Code': material['code'],
                'Item_Description': material['desc'],
                'Supplier': supplier['name'],
                'Purchase_Order_No': po_number,
                'Document_Number': doc_number,
                'Historical_Demand_Quantity (TON)': round(demand, 2),
                'Units_Sold/Used (TON)': round(units_used, 2),
                'Current_Stock_Level (TON)': round(current_stock, 2),
                'Safety_Stock_Target (TON)': round(safety_stock, 2),
                'Material_Type': material['type'],
                'Voltage_Level': material['voltage'],
                'Supplier_Country_Of_Origin': supplier['country'],
                'Price_Per_Unit (USD)': round(price, 2),
                'Delivery_Lead_Time (Days)': lead_time,
                'Payment_Terms': get_payment_terms(material['category'], supplier['performance']),
                'Market_Trends (Index)': round(market_trend, 1),
                'Number_Of_Alternative_Suppliers': num_alt_suppliers,
                'Supplier_Performance_Score (1-10)': supplier['performance'],
                'Risk_Score (1-10)': risk_score,
            }

            data.append(record)

        # Move to next month
        current_date += timedelta(days=32)
        current_date = current_date.replace(day=1)

    # Create DataFrame
    df = pd.DataFrame(data)

    # Print statistics
    print(f"\nDataset Generated Successfully!")
    print(f"Total Records: {len(df):,}")
    print(f"Date Range: {df['Date'].min()} to {df['Date'].max()}")
    print(f"\nMaterials Breakdown:")
    print(df['Material_Type'].value_counts().to_string())
    print(f"\nDemand Statistics (TON):")
    print(df['Historical_Demand_Quantity (TON)'].describe().to_string())
    print(f"\nTop Suppliers:")
    print(df['Supplier'].value_counts().head().to_string())

    return df

# ============================================================================
# EXECUTION
# ============================================================================

# Generate dataset
df = generate_dataset()

# Save to CSV
output_file = 'elsewedy_materials_dataset.csv'
df.to_csv(output_file, index=False)
print(f"\nDataset saved to: {output_file}")

# Show sample
print(f"\nSample Records (first 5):")
print(df.head().to_string())

Generating Elsewedy Cable Materials Dataset...
Date Range: 2022-01-01 to 2024-12-31
Materials: 17

Dataset Generated Successfully!
Total Records: 612
Date Range: 2022-01-01 to 2024-12-01

Materials Breakdown:
Material_Type
XLPE                   108
PE                      72
PVC                     72
GSW                     72
Semiconductor           72
GST                     36
LSF                     36
Copper Screen           36
Aluminum Screen         36
Mica Tape               36
Water Blocking Tape     36

Demand Statistics (TON):
count    612.000000
mean      27.633611
std       13.821946
min        6.580000
25%       18.147500
50%       25.285000
75%       35.662500
max       92.780000

Top Suppliers:
Supplier
MetalShield Corp        128
PolyTech Europe         109
TurkCable Materials      87
JapanSemi Advanced       69
Nordic Cables Supply     67

Dataset saved to: elsewedy_materials_dataset.csv

Sample Records (first 5):
         Date      Item_Code              Item_Descr

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# ============================================================================
# CONFIGURATION - EXPANDED DATASET
# ============================================================================

START_DATE = '2019-01-01'  # Extended from 2022 to 2019 (5 years)
END_DATE = '2024-12-31'
XLPE_GROWTH_RATE = 0.20
SEMICONDUCTOR_GROWTH_RATE = 0.19
DEFAULT_GROWTH_RATE = 0.10

# ============================================================================
# EXPANDED MATERIALS (30+ materials instead of 17)
# ============================================================================

MATERIALS = [
    # ============ XLPE - Multiple voltage levels ============
    {'code': 'XLPE-0.6KV-INS', 'desc': 'XLPE Insulation 0.6KV', 'type': 'XLPE', 'voltage': '0.6KV',
     'base_demand': 52, 'growth': 0.22, 'seasonal': True, 'category': 'B'},
    {'code': 'XLPE-11KV-INS', 'desc': 'XLPE Insulation 11KV', 'type': 'XLPE', 'voltage': '11KV',
     'base_demand': 45, 'growth': 0.20, 'seasonal': True, 'category': 'B'},
    {'code': 'XLPE-22KV-INS', 'desc': 'XLPE Insulation 22KV', 'type': 'XLPE', 'voltage': '22KV',
     'base_demand': 38, 'growth': 0.19, 'seasonal': True, 'category': 'B'},
    {'code': 'XLPE-33KV-INS', 'desc': 'XLPE Insulation 33KV', 'type': 'XLPE', 'voltage': '33KV',
     'base_demand': 35, 'growth': 0.18, 'seasonal': True, 'category': 'B'},
    {'code': 'XLPE-66KV-INS', 'desc': 'XLPE Insulation 66KV', 'type': 'XLPE', 'voltage': '66KV',
     'base_demand': 28, 'growth': 0.17, 'seasonal': True, 'category': 'B'},
    {'code': 'XLPE-132KV-INS', 'desc': 'XLPE Insulation 132KV', 'type': 'XLPE', 'voltage': '132KV',
     'base_demand': 22, 'growth': 0.16, 'seasonal': True, 'category': 'B'},

    # ============ PE - Various types ============
    {'code': 'PE-LV-JKT', 'desc': 'PE Jacket Low Voltage', 'type': 'PE', 'voltage': 'LV',
     'base_demand': 25, 'growth': 0.10, 'seasonal': False, 'category': 'B'},
    {'code': 'PE-MV-INS', 'desc': 'PE Insulation Medium Voltage', 'type': 'PE', 'voltage': 'MV',
     'base_demand': 20, 'growth': 0.09, 'seasonal': False, 'category': 'B'},
    {'code': 'PE-HV-JKT', 'desc': 'PE Jacket High Voltage', 'type': 'PE', 'voltage': 'HV',
     'base_demand': 18, 'growth': 0.11, 'seasonal': False, 'category': 'B'},
    {'code': 'PE-FOAM-LV', 'desc': 'Foam PE Low Voltage', 'type': 'PE', 'voltage': 'LV',
     'base_demand': 16, 'growth': 0.12, 'seasonal': False, 'category': 'B'},

    # ============ PVC - Multiple grades ============
    {'code': 'PVC-LV-INS-A', 'desc': 'PVC Insulation LV Grade A', 'type': 'PVC', 'voltage': 'LV',
     'base_demand': 30, 'growth': 0.08, 'seasonal': False, 'category': 'B'},
    {'code': 'PVC-LV-INS-B', 'desc': 'PVC Insulation LV Grade B', 'type': 'PVC', 'voltage': 'LV',
     'base_demand': 28, 'growth': 0.07, 'seasonal': False, 'category': 'B'},
    {'code': 'PVC-LV-JKT', 'desc': 'PVC Jacket Low Voltage', 'type': 'PVC', 'voltage': 'LV',
     'base_demand': 32, 'growth': 0.07, 'seasonal': False, 'category': 'B'},
    {'code': 'PVC-MV-INS', 'desc': 'PVC Insulation Medium Voltage', 'type': 'PVC', 'voltage': 'MV',
     'base_demand': 26, 'growth': 0.09, 'seasonal': False, 'category': 'B'},

    # ============ LSF - Fire resistant ============
    {'code': 'LSF-LV-JKT', 'desc': 'LSF Jacket Low Voltage', 'type': 'LSF', 'voltage': 'LV',
     'base_demand': 15, 'growth': 0.12, 'seasonal': False, 'category': 'B'},
    {'code': 'LSF-MV-JKT', 'desc': 'LSF Jacket Medium Voltage', 'type': 'LSF', 'voltage': 'MV',
     'base_demand': 12, 'growth': 0.13, 'seasonal': False, 'category': 'B'},
    {'code': 'LSF-HV-JKT', 'desc': 'LSF Jacket High Voltage', 'type': 'LSF', 'voltage': 'HV',
     'base_demand': 10, 'growth': 0.14, 'seasonal': False, 'category': 'B'},

    # ============ GSW - Galvanized Steel Wire ============
    {'code': 'GSW-MV-SHLD', 'desc': 'Galvanized Steel Wire Shield MV', 'type': 'GSW', 'voltage': 'MV',
     'base_demand': 20, 'growth': 0.10, 'seasonal': False, 'category': 'B'},
    {'code': 'GSW-HV-SHLD', 'desc': 'Galvanized Steel Wire Shield HV', 'type': 'GSW', 'voltage': 'HV',
     'base_demand': 16, 'growth': 0.09, 'seasonal': False, 'category': 'B'},
    {'code': 'GSW-EHV-SHLD', 'desc': 'Galvanized Steel Wire Shield EHV', 'type': 'GSW', 'voltage': 'EHV',
     'base_demand': 14, 'growth': 0.11, 'seasonal': False, 'category': 'B'},

    # ============ GST - Galvanized Steel Tape ============
    {'code': 'GST-MV-TAPE', 'desc': 'Galvanized Steel Tape MV', 'type': 'GST', 'voltage': 'MV',
     'base_demand': 19, 'growth': 0.10, 'seasonal': False, 'category': 'B'},
    {'code': 'GST-HV-TAPE', 'desc': 'Galvanized Steel Tape HV', 'type': 'GST', 'voltage': 'HV',
     'base_demand': 18, 'growth': 0.11, 'seasonal': False, 'category': 'B'},

    # ============ Copper Screens ============
    {'code': 'CU-SCR-11KV', 'desc': 'Copper Screen Tape 11KV', 'type': 'Copper Screen', 'voltage': '11KV',
     'base_demand': 12, 'growth': 0.09, 'seasonal': False, 'category': 'A'},
    {'code': 'CU-SCR-33KV', 'desc': 'Copper Screen Tape 33KV', 'type': 'Copper Screen', 'voltage': '33KV',
     'base_demand': 10, 'growth': 0.08, 'seasonal': False, 'category': 'A'},
    {'code': 'CU-SCR-66KV', 'desc': 'Copper Screen Tape 66KV', 'type': 'Copper Screen', 'voltage': '66KV',
     'base_demand': 8, 'growth': 0.07, 'seasonal': False, 'category': 'A'},

    # ============ Aluminum Screens ============
    {'code': 'AL-SCR-11KV', 'desc': 'Aluminum Screen Tape 11KV', 'type': 'Aluminum Screen', 'voltage': '11KV',
     'base_demand': 11, 'growth': 0.09, 'seasonal': False, 'category': 'A'},
    {'code': 'AL-SCR-33KV', 'desc': 'Aluminum Screen Tape 33KV', 'type': 'Aluminum Screen', 'voltage': '33KV',
     'base_demand': 10, 'growth': 0.08, 'seasonal': False, 'category': 'A'},
    {'code': 'AL-SCR-66KV', 'desc': 'Aluminum Screen Tape 66KV', 'type': 'Aluminum Screen', 'voltage': '66KV',
     'base_demand': 7, 'growth': 0.07, 'seasonal': False, 'category': 'A'},

    # ============ Mica Tapes ============
    {'code': 'MICA-TAPE-MV', 'desc': 'Mica Tape Medium Voltage', 'type': 'Mica Tape', 'voltage': 'MV',
     'base_demand': 9, 'growth': 0.08, 'seasonal': False, 'category': 'B'},
    {'code': 'MICA-TAPE-HV', 'desc': 'Mica Tape High Voltage', 'type': 'Mica Tape', 'voltage': 'HV',
     'base_demand': 8, 'growth': 0.07, 'seasonal': False, 'category': 'B'},
    {'code': 'MICA-TAPE-EHV', 'desc': 'Mica Tape Extra High Voltage', 'type': 'Mica Tape', 'voltage': 'EHV',
     'base_demand': 6, 'growth': 0.06, 'seasonal': False, 'category': 'B'},

    # ============ Water Blocking Tapes ============
    {'code': 'WB-TAPE-LV', 'desc': 'Water Blocking Tape LV', 'type': 'Water Blocking Tape', 'voltage': 'LV',
     'base_demand': 24, 'growth': 0.14, 'seasonal': True, 'category': 'B'},
    {'code': 'WB-TAPE-MV', 'desc': 'Water Blocking Tape MV', 'type': 'Water Blocking Tape', 'voltage': 'MV',
     'base_demand': 22, 'growth': 0.13, 'seasonal': True, 'category': 'B'},
    {'code': 'WB-TAPE-HV', 'desc': 'Water Blocking Tape HV', 'type': 'Water Blocking Tape', 'voltage': 'HV',
     'base_demand': 18, 'growth': 0.12, 'seasonal': True, 'category': 'B'},

    # ============ Semiconductors ============
    {'code': 'SEMI-11KV-LAY', 'desc': 'Semiconductor Layer 11KV', 'type': 'Semiconductor', 'voltage': '11KV',
     'base_demand': 28, 'growth': 0.19, 'seasonal': True, 'category': 'B'},
    {'code': 'SEMI-22KV-LAY', 'desc': 'Semiconductor Layer 22KV', 'type': 'Semiconductor', 'voltage': '22KV',
     'base_demand': 26, 'growth': 0.18, 'seasonal': True, 'category': 'B'},
    {'code': 'SEMI-33KV-LAY', 'desc': 'Semiconductor Layer 33KV', 'type': 'Semiconductor', 'voltage': '33KV',
     'base_demand': 24, 'growth': 0.18, 'seasonal': True, 'category': 'B'},
    {'code': 'SEMI-66KV-LAY', 'desc': 'Semiconductor Layer 66KV', 'type': 'Semiconductor', 'voltage': '66KV',
     'base_demand': 20, 'growth': 0.17, 'seasonal': True, 'category': 'B'},
]

# Expanded supplier list
SUPPLIERS = [
    {'name': 'PolyTech Europe GmbH', 'country': 'Germany', 'specialty': ['XLPE', 'PE', 'PVC'], 'performance': 9, 'base_lead_time': 35},
    {'name': 'Nordic Polymers AB', 'country': 'Sweden', 'specialty': ['XLPE', 'Semiconductor', 'PE'], 'performance': 10, 'base_lead_time': 30},
    {'name': 'ChinaPoly Industries', 'country': 'China', 'specialty': ['PVC', 'PE', 'LSF'], 'performance': 7, 'base_lead_time': 45},
    {'name': 'Shanghai Cable Materials', 'country': 'China', 'specialty': ['XLPE', 'PVC', 'Water Blocking Tape'], 'performance': 8, 'base_lead_time': 42},
    {'name': 'TurkCable Materials Ltd', 'country': 'Turkey', 'specialty': ['PVC', 'LSF', 'GSW'], 'performance': 8, 'base_lead_time': 25},
    {'name': 'Ankara Insulation Tech', 'country': 'Turkey', 'specialty': ['PE', 'LSF', 'Water Blocking Tape'], 'performance': 7, 'base_lead_time': 28},
    {'name': 'MetalShield Korea', 'country': 'South Korea', 'specialty': ['GSW', 'GST', 'Copper Screen', 'Aluminum Screen'], 'performance': 9, 'base_lead_time': 40},
    {'name': 'Seoul Metal Works', 'country': 'South Korea', 'specialty': ['Copper Screen', 'Aluminum Screen', 'GSW'], 'performance': 9, 'base_lead_time': 38},
    {'name': 'IndiaScreen Technologies', 'country': 'India', 'specialty': ['Aluminum Screen', 'Mica Tape', 'Water Blocking Tape'], 'performance': 7, 'base_lead_time': 50},
    {'name': 'Mumbai Cable Supplies', 'country': 'India', 'specialty': ['PE', 'PVC', 'Mica Tape'], 'performance': 6, 'base_lead_time': 52},
    {'name': 'USATape Solutions Inc', 'country': 'USA', 'specialty': ['Mica Tape', 'Water Blocking Tape'], 'performance': 8, 'base_lead_time': 42},
    {'name': 'JapanSemi Advanced Materials', 'country': 'Japan', 'specialty': ['Semiconductor', 'XLPE'], 'performance': 10, 'base_lead_time': 38},
    {'name': 'Tokyo Polymer Industries', 'country': 'Japan', 'specialty': ['XLPE', 'PE', 'Semiconductor'], 'performance': 9, 'base_lead_time': 40},
    {'name': 'ItalCavi Materials SpA', 'country': 'Italy', 'specialty': ['XLPE', 'PVC', 'LSF'], 'performance': 9, 'base_lead_time': 33},
    {'name': 'FrenchPoly Solutions', 'country': 'France', 'specialty': ['XLPE', 'PE', 'LSF'], 'performance': 8, 'base_lead_time': 35},
]

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def get_suppliers_for_material(material_type):
    """Get suitable suppliers for a material type"""
    suitable = [s for s in SUPPLIERS if material_type in s['specialty']]
    return suitable if suitable else SUPPLIERS[:3]

def calculate_demand(base_demand, growth_rate, months_elapsed, seasonal, month, year):
    """Calculate demand with growth trend, seasonality, and random events"""
    # Base trend with growth
    trend = base_demand * (1 + growth_rate) ** (months_elapsed / 12)

    # Add seasonality (higher in Q2 and Q4 for infrastructure projects)
    if seasonal:
        seasonal_factor = 1 + 0.18 * np.sin(2 * np.pi * month / 12 + np.pi/2)
        trend *= seasonal_factor

    # Add yearly patterns (some years have more projects)
    if year in [2020, 2023]:  # Example: COVID dip, then recovery
        trend *= 0.85
    elif year in [2021, 2024]:
        trend *= 1.10

    # Add random noise (±12%)
    noise = np.random.normal(1, 0.12)
    demand = max(0, trend * noise)

    return round(demand, 2)

def calculate_safety_stock(avg_demand, lead_time, risk_score):
    """Calculate safety stock based on demand variability and risk"""
    risk_multiplier = 1 + (risk_score / 10) * 0.5
    lead_time_demand = avg_demand * (lead_time / 30)
    safety_stock = lead_time_demand * risk_multiplier
    return round(safety_stock, 2)

def calculate_risk_score(num_suppliers, lead_time, performance_score):
    """Calculate supply chain risk score (1-10, higher = more risk)"""
    supplier_risk = max(1, 11 - num_suppliers * 2)
    lead_time_risk = min(10, lead_time / 7)
    performance_risk = 11 - performance_score
    risk = (supplier_risk * 0.4 + lead_time_risk * 0.3 + performance_risk * 0.3)
    return round(np.clip(risk, 1, 10), 1)

def get_payment_terms(category, supplier_performance):
    """Get payment terms based on material category and supplier relationship"""
    terms_options = {
        'A': ['Net 90', 'Net 120', 'Annual Contract'],
        'B': ['Net 30', 'Net 45', 'Net 60'],
        'C': ['Net 30', 'Net 45']
    }

    if supplier_performance >= 9:
        return random.choice(terms_options.get(category, ['Net 30']))
    else:
        return random.choice(terms_options.get(category, ['Net 30'])[:2])

# ============================================================================
# MAIN DATASET GENERATION
# ============================================================================

def generate_dataset():
    """Generate complete synthetic dataset"""

    print("Generating EXPANDED Elsewedy Cable Materials Dataset...")
    print(f"Date Range: {START_DATE} to {END_DATE} (5 years)")
    print(f"Materials: {len(MATERIALS)}")
    print(f"Suppliers: {len(SUPPLIERS)}")

    start = datetime.strptime(START_DATE, '%Y-%m-%d')
    end = datetime.strptime(END_DATE, '%Y-%m-%d')

    data = []
    current_date = start

    while current_date <= end:
        month_num = current_date.month
        year_num = current_date.year
        months_elapsed = (current_date.year - start.year) * 12 + (current_date.month - start.month)

        for material in MATERIALS:
            # Get suppliers
            suitable_suppliers = get_suppliers_for_material(material['type'])
            num_alt_suppliers = len(suitable_suppliers)

            # Select primary supplier
            weights = [s['performance'] for s in suitable_suppliers]
            supplier = random.choices(suitable_suppliers, weights=weights, k=1)[0]

            # Calculate demand
            demand = calculate_demand(
                material['base_demand'],
                material['growth'],
                months_elapsed,
                material['seasonal'],
                month_num,
                year_num
            )

            # Lead time with variability
            lead_time = supplier['base_lead_time'] + random.randint(-5, 5)

            # Risk score
            risk_score = calculate_risk_score(
                num_alt_suppliers,
                lead_time,
                supplier['performance']
            )

            # Safety stock
            safety_stock = calculate_safety_stock(
                material['base_demand'],
                lead_time,
                risk_score
            )

            # Current stock
            stock_target = demand * random.uniform(2, 3.5)
            current_stock = stock_target + np.random.normal(0, stock_target * 0.18)
            current_stock = max(0, current_stock)

            # Units used
            units_used = demand * random.uniform(0.80, 1.05)

            # Price
            base_prices = {
                'XLPE': 3.5, 'PE': 2.8, 'PVC': 2.2, 'LSF': 3.0,
                'GSW': 4.2, 'GST': 3.8, 'Copper Screen': 8.5, 'Aluminum Screen': 5.5,
                'Mica Tape': 6.0, 'Water Blocking Tape': 2.5, 'Semiconductor': 7.5
            }
            base_price = base_prices.get(material['type'], 3.0)

            # Add price trends over years
            price_trend = 1.0
            if year_num >= 2020:
                price_trend *= (1.03 ** (year_num - 2020))  # 3% inflation per year

            price = base_price * price_trend * (1 + np.random.normal(0, 0.08))

            # Market trends
            market_trend = 100 + 10 * np.sin(2 * np.pi * month_num / 12) + np.random.normal(0, 5)
            if year_num in [2020]:  # COVID impact
                market_trend *= 0.85

            # Generate PO
            po_number = f"PO-{year_num}-{material['code'][:4]}-{random.randint(1000,9999)}"
            doc_number = f"DOC-{random.randint(100000,999999)}"

            record = {
                'Date': current_date.strftime('%Y-%m-%d'),
                'Item_Code': material['code'],
                'Item_Description': material['desc'],
                'Supplier': supplier['name'],
                'Purchase_Order_No': po_number,
                'Document_Number': doc_number,
                'Historical_Demand_Quantity (TON)': round(demand, 2),
                'Units_Sold/Used (TON)': round(units_used, 2),
                'Current_Stock_Level (TON)': round(current_stock, 2),
                'Safety_Stock_Target (TON)': round(safety_stock, 2),
                'Material_Type': material['type'],
                'Voltage_Level': material['voltage'],
                'Supplier_Country_Of_Origin': supplier['country'],
                'Price_Per_Unit (USD)': round(price, 2),
                'Delivery_Lead_Time (Days)': lead_time,
                'Payment_Terms': get_payment_terms(material['category'], supplier['performance']),
                'Market_Trends (Index)': round(market_trend, 1),
                'Number_Of_Alternative_Suppliers': num_alt_suppliers,
                'Supplier_Performance_Score (1-10)': supplier['performance'],
                'Risk_Score (1-10)': risk_score,
            }

            data.append(record)

        # Move to next month
        current_date += timedelta(days=32)
        current_date = current_date.replace(day=1)

    df = pd.DataFrame(data)

    # Print statistics
    print(f"\n Dataset Generated Successfully!")
    print(f" Total Records: {len(df):,}")
    print(f" Date Range: {df['Date'].min()} to {df['Date'].max()}")
    print(f" Unique Materials: {df['Item_Code'].nunique()}")
    print(f"\n Materials Breakdown:")
    print(df['Material_Type'].value_counts().to_string())
    print(f"\n Demand Statistics (TON):")
    print(df['Historical_Demand_Quantity (TON)'].describe().to_string())
    print(f"\n Suppliers:")
    print(df['Supplier'].value_counts().to_string())

    return df

# ============================================================================
# EXECUTION
# ============================================================================

df = generate_dataset()

output_file = 'elsewedy_materials_dataset_expanded.csv'
df.to_csv(output_file, index=False)
print(f"\n Dataset saved to: {output_file}")

print(f"\nSample Records:")
print(df.head(10).to_string())

print("\n" + "="*70)
print("EXPANDED DATASET READY!")
print("="*70)
print(f"\nDataset Size: {len(df):,} records")
print(f"   • {len(df) / df['Item_Code'].nunique():.0f} records per material")
print(f"   • Perfect for Deep Learning! 🚀")
print("\nThis dataset is {len(df) / 600:.1f}x larger than before!")

Generating EXPANDED Elsewedy Cable Materials Dataset...
Date Range: 2019-01-01 to 2024-12-31 (5 years)
Materials: 38
Suppliers: 15

 Dataset Generated Successfully!
 Total Records: 2,736
 Date Range: 2019-01-01 to 2024-12-01
 Unique Materials: 38

 Materials Breakdown:
Material_Type
XLPE                   432
PE                     288
PVC                    288
Semiconductor          288
LSF                    216
Copper Screen          216
GSW                    216
Mica Tape              216
Aluminum Screen        216
Water Blocking Tape    216
GST                    144

 Demand Statistics (TON):
count    2736.000000
mean       30.653319
std        23.397787
min         4.090000
25%        15.247500
50%        25.125000
75%        37.675000
max       220.330000

 Suppliers:
Supplier
MetalShield Korea               404
Seoul Metal Works               270
Tokyo Polymer Industries        202
Nordic Polymers AB              198
PolyTech Europe GmbH            173
IndiaScreen Technologi